<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 220px; height: 150px; vertical-align: middle;">
            <img src="../assets/aaa.png" width="220" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Autonomous Traders</h2>
            <span style="color:#ff7800;">MCP サーバーのツールとリソースによって動かされる自律型エージェントを示すための、株式取引シミュレーションです。
            </span>
        </td>
    </tr>
</table>

### Week 6 Day 5

## 改善1: 可観測性

#### ログの裏にあるカスタムトレーサー

各パネルには、トレーダーの活動をリアルタイムで表示するログがあり、ステップの種類(エージェント、関数呼び出し、生成、応答、アカウントイベント)によって色分けされています。このログは、私たち自身の仕組みです。OpenAI Agents SDK ではそのトレーシング機構に接続できるので、`backend/tracers.py` は `LogTracer` を定義しています。これは `TracingProcessor` で、トレーダーが実行されるたびにすべてのスパンをデータベースに書き込みます。これは、前回のラボで見た `add_trace_processor(LogTracer())` によって一度だけ登録されます。API はそれらの行を読み返し、フロントエンドがそれらに色を付けるので、画面を流れていく思考の様子は、まさに SDK のトレースストリームから、自分たちのプロセッサによって取り込まれたものなのです。

どのように動いているか、`backend/tracers.py` を見てみてください。

## 改善2: 評価とフィードバック

#### 自ら改善していく戦略

各トレーダーの戦略は、そのパネルの上部に表示されています。トレーダーはその戦略に固定されているわけではありません。`change_strategy` は `backend/accounts_server.py` で定義されているトレーダーのツールの1つで、彼らの instructions では、実際の取引の成果を振り返り、そこから得た教訓を戦略に反映させるよう求めています。トレーダーがこれを行うと、その戦略のテキストが変化し、ログに「Changed strategy」というエントリが現れます。

これは、エージェント型 AI におけるもっとも重要な考え方の1つであるフィードバックループを完成させます。ここでいう現実世界の評価とは、ポートフォリオのリターンのことで、それが次回のエージェントの判断にフィードバックされます。エージェントは自分自身の行動の結果を使って instructions を書き直し、時間とともに改善していきます。

## 改善3: 本番用フロントエンド

前回のラボでは、ダッシュボードは Gradio アプリでした。それは Python の中で動き、取引用データベースを直接読んでいたので、素早く構築でき、社内ツールとしては理想的です。本番システムは通常、別の形で構成されます。バックエンドが自分のデータを HTTP API として公開し、別のフロントエンドの Web アプリがその API を利用するのです。この最後のラボでは、まさにそれを私たちの取引フロアに対して行います。トレーダーとデータベースは変わりません。その前に小さな FastAPI 層を追加し、その上に Vite と TypeScript のフロントエンドを載せます。

## どう分割されているか

すでにある `backend` パッケージの隣に、新しい部品が2つ加わります。

`backend/api.py` は薄い FastAPI アプリです。Gradio ダッシュボードが読んでいたのと同じ accounts とログを読み、それらを JSON として返します。トレーダーの一覧、各トレーダーのポートフォリオ価値、利益、保有株、取引、ライブの活動ログ、そしてどの市場データソースが動いているか、などです。トレーダー自体を実行するわけではありません。取引フロアのエンジンは、依然として独立してそれを行います。

`frontend/` は Vite と TypeScript のアプリです。2秒おきに API を呼び出し、4人のトレーダーをグリッドに描画します。それぞれにライブのポートフォリオチャート、保有株のヒートマップ、そしてあなたのカスタムトレーシングによる活動ログが表示されます。ダークテーマとライトテーマがあり、価格がシミュレートされているかライブかを示すバッジもあります。

この2つは HTTP でしかやり取りしないので、別々にホストすることも、同じ API の前に別のクライアントを置くこともできます。開発時には、Vite サーバーが `/api` をポート8000の FastAPI バックエンドにプロキシするので、ブラウザからは単一のオリジンに見え、CORS の設定は不要です。

API 全体は、ほんの一握りの読み取り専用エンドポイントにすぎません。以下がそれらです。

In [ ]:
from backend.api import app

for route in app.routes:
    methods = getattr(route, "methods", None)
    if methods and route.path.startswith("/api"):
        print(sorted(methods), route.path)

## 一度だけのセットアップ

フロントエンドは Node のプロジェクトなので、依存関係を一度だけインストールしてください。Node は、`npx` を使った以前のラボで、すでにあなたのマシンにインストールされています。

`cd 6_mcp/frontend`

`npm install`

## 実行してみよう

ターミナルパネルの「+」で開いた、3つのターミナルを使います。

まず、API を起動します。

`cd 6_mcp`

`uv run uvicorn backend.api:app --port 8000`

自分でエンドポイントを見てみたい場合は、FastAPI が http://localhost:8000/docs でインタラクティブなドキュメントも提供しています。

次に、フロントエンドを起動します。

`cd 6_mcp/frontend`

`npm run dev`

http://localhost:5173 を開いてください。4人のトレーダーがすぐに現れ、API から読み込まれます。角にあるテーマの切り替えを試してみて、サイドバーの市場データバッジにも注目してください。

最後に、取引フロアのエンジンを起動して、トレーダーたちが動き出す様子を見てみましょう。

`cd 6_mcp`

`uv run -m backend.trading_floor`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">最後にもう一度、API の使用量に注意してください</h2>
            <span style="color:#ff7800;">このエンジンは、1時間ごと、あるいは設定した RUN_EVERY_N_MINUTES の間隔でループを続けます。API の使用量に注意し、十分見終わったら止めてください。私はこれを何時間も楽しく眺めてしまいがちですが、あなたもそうなることを願っています。
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">本当に、本当にありがとうございました!</h2>
            <span style="color:#00cc00;">これで完結です! ここまで最後までやり遂げてくれたことに、心から感謝します。編集者にまた叱られそうですが、もう一度お伝えしておきます。もし Udemy でこのコースを評価していただけたら、本当にありがたいです。Udemy がこのコースを他の人にも表示するかどうかを判断する、もっとも重要な材料となり、大きな違いを生みます。<br/><br/>ぜひ連絡を取り続けて、その後どうなったか教えてください。そして、この驚くべき時代における、この驚くべき分野でのあなたの旅を、ぜひ共有してください。もしここまで読んでくれて、それでも LinkedIn でつながることをまだ避けているなら - <a href="https://www.linkedin.com/in/eddonner/">もう一度、ここにいます</a>! このコースでの成果を投稿したい場合は、私をタグ付けしてください。反応して、あなたの発信を広めるお手伝いをします。<br><br/>もう少し私に付き合ってもいいという方は、
            <a href="https://edwarddonner.com/curriculum">私の全カリキュラムをぜひご覧ください</a>。Proficient AI Engineer として認定される方法など、さらに多くのことが載っています。<br/><br/>もう一度言います -- おめでとうございます!
            </span>
        </td>
    </tr>
</table>